# 07 Land capacity

Reports the City of Revelstoke's real zoning distribution (by parcel count) and the Resort Lands DPA's area, both queried live from the City's ArcGIS FeatureServers. Does not compute a unit-capacity number: that needs the zoning bylaw's density provisions (units per lot, FAR, minimum lot size per zone), which have not been found as a document this session (see Open issues in steps/07). What this notebook can show is which zones even allow multi-unit housing, and how many parcels carry them.

In [ ]:
research_dir = "research"
processed_dir = "data/processed"
raw_dir = "data/raw"
project_crs = "EPSG:26911"
dem_clip_buffer_m = 200

# Filter 5 (service distance): no source states what counts as "serviced";
# tested across all three, one carried forward as the primary scenario.
service_distances_m = [50, 100, 200]
primary_service_distance_m = 100

# Filter 6 (building coverage / "underused"): same treatment.
building_coverage_cutoffs = [0.10, 0.20, 0.30]
primary_building_coverage_cutoff = 0.20

# Filter 7 (DEM slope): same treatment.
slope_cutoffs_pct = [15, 20, 25, 30]
primary_slope_cutoff_pct = 20

## Zoning distribution (transcribed from claim C049)

Queried live this session via the FeatureServer's statistics endpoint (group by zoningName, count of OBJECTID). Kept as a literal transcription here, same reasoning as notebooks 02/03: one place records what was queried, this notebook works from the recorded result.

In [ ]:
ZONING_COUNTS = """
zoning_code,zoning_name,parcel_count
P-3,Public Utilities Zone,63
MU-1,Downtown Zone,257
R-LD5,Manufactured Home and Ground-Oriented Dwelling Zone,102
R-MD2,Multi-Unit Apartment Zone,6
CD-02,Comprehensive Development Zone 2,93
I-1,Low Impact Zone,63
CD-09,Comprehensive Development Zone 9,1
R-LD4,Manufactured Home Zone,26
R-LD3,Multi-Unit Rowhouse Zone,7
I-2,Medium Impact Zone,12
CD-01,Comprehensive Development Zone 1,2
I-4,Airport Zone,1
MU-2,Downtown Fringe Zone,96
C-2,Urban Tourist Accommodation Zone,22
R-LD6,Tourist Accommodation Zone,94
CD-05,Comprehensive Development Zone 5,2
P-2,Institutional Zone,89
CD-06,Comprehensive Development Zone 6,6
I-5,Rural Recreation and Natural Resource Zone,26
C-1,Highway Commercial Zone,30
R-LD7,Rural Residential Zone,40
C-4,Local Neighbourhood Commercial Zone,7
CD-10,Comprehensive Development Zone 10,27
R-LD2,Small Lot Ground-Oriented Dwelling Zone,85
CD-03,Comprehensive Development Zone 3,9
CD-12,Comprehensive Development Zone 12,4
CD-11,Comprehensive Development Zone 11,11
C-3,Rural Tourist Accommodation Zone,3
CD-04,Comprehensive Development Zone 4,1
R-LD1,Ground-Oriented Dwelling Zone,2891
E-1,Environmental and Rural Recreation Zone,301
R-MD3,Multi-Unit Condominium Zone,47
CD-07,Comprehensive Development Zone 7,1
I-3,High Impact Zone,56
P-1,Parks and Public Use Zone,213
MU-3,Corridor Zone,53
R-MD1,Multi-Unit Rowhouse Zone,10
CD-08,Comprehensive Development Zone 8,5
MU-4,Live-Work Zone,51
"""

In [ ]:
import io
import sys

import pandas as pd

sys.path.insert(0, "src")
from resort.ledger import read_ledger

ledger = read_ledger(
    claims_path=f"{research_dir}/claims.csv",
    sources_path=f"{research_dir}/sources.csv",
)
assert ledger["claims"]["C049"]["source_id"] == "S045"

zoning = pd.read_csv(io.StringIO(ZONING_COUNTS))
assert zoning["parcel_count"].sum() > 0
zoning.sort_values("parcel_count", ascending=False).head(10)

## Which zones allow multi-unit housing at all

A zone's own name states its use type (rowhouse, apartment, condominium, downtown/corridor/live-work mixed use); this is evidence, read directly off the zone names, not a judgment about density. How many units each such parcel could actually hold is not answered here.

In [ ]:
MULTI_UNIT_ELIGIBLE_PREFIXES = ("R-MD", "R-LD3", "MU-")

In [ ]:
zoning["multi_unit_eligible_by_name"] = zoning["zoning_code"].str.startswith(
    MULTI_UNIT_ELIGIBLE_PREFIXES
)
multi_unit_summary = (
    zoning.groupby("multi_unit_eligible_by_name")["parcel_count"].sum()
)
multi_unit_summary

## Resort Lands DPA area (claim C050)

One polygon, one area figure; kept as plain values rather than a table.

In [ ]:
assert ledger["claims"]["C050"]["source_id"] == "S047"
resort_lands_dpa = {
    "area_sq_m": 8110956.33984375,
    "area_hectares": 8110956.33984375 / 10_000,
}
resort_lands_dpa

## Write outputs

Zoning table and the resort-lands area, so step 08 can cite the same numbers without re-querying the FeatureServers.

In [ ]:
import json
import os

os.makedirs(processed_dir, exist_ok=True)
zoning.to_csv(f"{processed_dir}/07_zoning_by_parcel_count.csv", index=False, encoding="utf-8")
with open(f"{processed_dir}/07_resort_lands_dpa.json", "w", encoding="utf-8") as f:
    json.dump(resort_lands_dpa, f, indent=2)
print("wrote 07_zoning_by_parcel_count.csv, 07_resort_lands_dpa.json")

## Checks

Every parcel must be classified into exactly one of the two multi-unit-eligible groups (no zone left unclassified), and the total parcel count from the grouped sum must match the raw table's own sum, since a mismatch would mean a zone code was silently dropped or double-counted.

In [ ]:
assert multi_unit_summary.sum() == zoning["parcel_count"].sum()
assert zoning["multi_unit_eligible_by_name"].notna().all()
print("checks passed")

# Spatial parcel-suitability funnel (steps/07's proposed method)

Builds the funnel proposed in steps/07: real ParcelMap BC parcel geometry, filtered by zoning eligibility, floodplain/hazardous-DPA/ALR exclusion, service distance, building coverage, and DEM slope -- reporting count and area at every stage, never only the final number. Three of the seven filters are unsourced judgment calls, each sensitivity-tested across a stated range rather than picking one value silently. Unit capacity is not computed: research/inbox/ has no zoning bylaw density extraction, so it stays absent rather than estimated.

## Confirm the claims this section rests on

C049/C065 are the zoning and parcel data already used above; C056-C062 are the constraint/service layers; C071-C073 are new findings from actually querying them this session (see notes in research/claims.csv).

In [ ]:
sys.path.insert(0, "src")
from resort.arcgis import fetch_city_layer, fetch_parcelmap_bc
from resort.dem import clip_dem, compute_slope_aspect, sample_at_point

for claim_id, source_id in [
    ("C049", "S045"), ("C065", "S030"), ("C056", "S052"), ("C057", "S053"),
    ("C058", "S054"), ("C059", "S055"), ("C060", "S056"), ("C061", "S057"),
    ("C062", "S058"), ("C071", "S030"), ("C072", "S050"), ("C073", "S054"), ("C074", "S064"),
]:
    assert ledger["claims"][claim_id]["source_id"] == source_id
print("all claims confirmed against their sources")

## Base geometry: ParcelMap BC parcels in the City of Revelstoke

Filtered by ParcelMap BC's own MUNICIPALITY attribute (C065), not a separately-sourced boundary polygon. A small number of returned parcels sit far outside the zoned area (C071, cross-checked against the live count in the code cell below) and are excluded from the funnel's working set, reported here rather than silently dropped.

In [ ]:
parcels_raw = fetch_parcelmap_bc("MUNICIPALITY='Revelstoke, City of'", out_crs=project_crs)
parcel_bounds = parcels_raw.geometry.bounds
outlier_mask = parcel_bounds["miny"] < 5_600_000
parcels_outliers = parcels_raw[outlier_mask]
parcels = parcels_raw[~outlier_mask].reset_index(drop=True)
parcels["parcel_area_sqm"] = parcels.geometry.area
print(f"parcels_raw: {len(parcels_raw)}, excluded as outliers: {len(parcels_outliers)}, "
      f"working set: {len(parcels)}")
assert len(parcels_outliers) == 2

## Funnel stage 0: starting point

Every later stage reports count and area relative to this.

In [ ]:
funnel_rows = [{
    "stage": "0. all City of Revelstoke parcels (ParcelMap BC)",
    "parcel_count": len(parcels), "area_ha": round(parcels["parcel_area_sqm"].sum() / 10_000, 2),
}]
funnel_rows[-1]

## Filter 1: zoning (multi-unit eligible by name)

Reuses the MULTI_UNIT_ELIGIBLE_PREFIXES classification already defined above. Each parcel is assigned the zone of the zoning polygon containing its representative point (a point guaranteed to be inside the parcel, unlike a centroid which can fall outside for concave shapes).

In [ ]:
import geopandas as gpd

zoning_live = fetch_city_layer("Zoning_(View)", out_sr=int(project_crs.split(":")[1]))
zoning_live["zoning_code"] = zoning_live["zoningName"].str.split(" - ").str[0]

parcels["rep_point"] = parcels.geometry.representative_point()
parcels_pts = parcels.set_geometry("rep_point")
joined = gpd.sjoin(parcels_pts, zoning_live[["zoning_code", "geometry"]], how="left", predicate="within")
ambiguous_parcels = joined.index[joined.index.duplicated()].unique()
print(f"parcels whose representative point falls inside 2+ overlapping zoning polygons: "
      f"{len(ambiguous_parcels)} (kept the first match for each; overlap is a real quirk in "
      f"the Zoning layer's own geometry, not a join bug)")
joined_first = joined[~joined.index.duplicated(keep="first")]
parcels["zoning_code"] = joined_first["zoning_code"].reindex(parcels.index)
parcels = parcels.drop(columns="rep_point")

parcels["multi_unit_eligible_by_name"] = parcels["zoning_code"].str.startswith(
    MULTI_UNIT_ELIGIBLE_PREFIXES
).fillna(False)
eligible = parcels[parcels["multi_unit_eligible_by_name"]].copy()
print(f"unmatched (no zoning polygon found for parcel's representative point): "
      f"{parcels['zoning_code'].isna().sum()}")
funnel_rows.append({
    "stage": "1. zoned multi-unit eligible",
    "parcel_count": len(eligible), "area_ha": round(eligible["parcel_area_sqm"].sum() / 10_000, 2),
})
funnel_rows[-1]

## Working extent for the remaining filters

From here on, only zoning-eligible parcels' own bounding box is used to scope further live queries (floodplain, hazardous DPA, ALR, buildings) -- these filters only matter for parcels that already passed Filter 1, and the hazardous-DPA layer in particular is expensive to page through (C073), so querying only the relevant extent instead of the whole municipality is a real saving, not just a convenience.

In [ ]:
work_bounds = tuple(eligible.total_bounds)
print(f"working extent: {work_bounds}")
print(f"extent size: {(work_bounds[2]-work_bounds[0])/1000:.1f} x {(work_bounds[3]-work_bounds[1])/1000:.1f} km")

## Filters 2-4: floodplain, hazardous DPA, ALR

Each is a real regulatory/environmental layer (C057-C059); a parcel intersecting any part of one is excluded. The hazardous DPA layer is ~1 sq m raster-derived cells (C073), so it is dissolved into one geometry before the intersection test.

In [ ]:
floodplain = fetch_city_layer("OCP_Floodplain_2_view", envelope=work_bounds)
hazard = fetch_city_layer("OCP_Environmentally_Hazardous_DPA_view", envelope=work_bounds)
alr = fetch_city_layer("OCP_Agricultural_Land_Reserve_(ALR)", envelope=work_bounds)

floodplain_union = floodplain.union_all() if len(floodplain) else None
hazard_union = hazard.union_all() if len(hazard) else None
alr_union = alr.union_all() if len(alr) else None

survivors = eligible.copy()
for label, union_geom in [
    ("2. outside floodplain", floodplain_union),
    ("3. outside hazardous DPA", hazard_union),
    ("4. outside ALR", alr_union),
]:
    if union_geom is not None:
        survivors = survivors[~survivors.geometry.intersects(union_geom)]
    funnel_rows.append({
        "stage": label, "parcel_count": len(survivors),
        "area_ha": round(survivors["parcel_area_sqm"].sum() / 10_000, 2),
    })
funnel_rows[-3:]

## Filter 5: service distance (judgment call, sensitivity-tested)

No source states what counts as "serviced"; tested across three candidate distances (set in the parameters cell) from both Water Mains and Sanitary Mains (a parcel needs to be near both to be realistically serviceable, not just one).

In [ ]:
water = fetch_city_layer("Water_Mains_view", envelope=work_bounds)
sanitary = fetch_city_layer("Sanitary_Mains_view", envelope=work_bounds)
water_union = water.union_all()
sanitary_union = sanitary.union_all()

service_sensitivity_rows = []
service_survivors_by_distance = {}
for dist in service_distances_m:
    near_both = survivors[
        survivors.geometry.intersects(water_union.buffer(dist))
        & survivors.geometry.intersects(sanitary_union.buffer(dist))
    ]
    service_survivors_by_distance[dist] = near_both
    service_sensitivity_rows.append({
        "filter": "service_distance_m", "value": dist,
        "parcel_count": len(near_both), "area_ha": round(near_both["parcel_area_sqm"].sum() / 10_000, 2),
    })
pd.DataFrame(service_sensitivity_rows)

## Filter 6: building coverage / "underused" (judgment call, sensitivity-tested)

Building footprint area over parcel area, under a stated cutoff (low coverage = plausibly underused). No source states this cutoff either; tested across three candidate cutoffs (set in the parameters cell), applied on top of the primary service-distance survivors from Filter 5.

In [ ]:
buildings = fetch_city_layer("Buildings_view", envelope=work_bounds)

base_for_coverage = service_survivors_by_distance[primary_service_distance_m].copy()
# gpd.overlay clips each building to its actual intersection with each parcel, so a building
# straddling two parcels only credits each parcel its own real overlap area, not the whole
# building's footprint (which sjoin + a plain area-sum would have done).
overlap = gpd.overlay(
    buildings[["geometry"]], base_for_coverage[["geometry"]].reset_index(), how="intersection"
)
overlap["overlap_area_sqm"] = overlap.geometry.area
bldg_area_by_parcel = overlap.groupby("index")["overlap_area_sqm"].sum()
base_for_coverage["building_area_sqm"] = base_for_coverage.index.map(bldg_area_by_parcel).fillna(0)
base_for_coverage["building_coverage"] = base_for_coverage["building_area_sqm"] / base_for_coverage["parcel_area_sqm"]

coverage_sensitivity_rows = []
coverage_survivors_by_cutoff = {}
for cutoff in building_coverage_cutoffs:
    under = base_for_coverage[base_for_coverage["building_coverage"] < cutoff]
    coverage_survivors_by_cutoff[cutoff] = under
    coverage_sensitivity_rows.append({
        "filter": "building_coverage_cutoff", "value": cutoff,
        "parcel_count": len(under), "area_ha": round(under["parcel_area_sqm"].sum() / 10_000, 2),
    })
pd.DataFrame(coverage_sensitivity_rows)

## Filter 7: DEM slope (judgment call, sensitivity-tested)

Mean DEM slope per parcel, under a stated cutoff (candidate values in the parameters cell). Clipped only to the parcels still in play after Filters 1-6 (much smaller than the full municipality), which is also simply the correct place for this filter to sit in the funnel's own declared order.

In [ ]:
base_for_slope = coverage_survivors_by_cutoff[primary_building_coverage_cutoff]
slope_bounds = tuple(base_for_slope.total_bounds)
dem = clip_dem(slope_bounds, project_crs, buffer_m=dem_clip_buffer_m)
slope_deg, slope_pct, _ = compute_slope_aspect(dem)
dem_transform = dem.rio.transform()
print(f"DEM shape {slope_pct.shape} over the filter-7 working extent")

## Per-parcel mean slope

Each parcel gets a small window sliced from the already-clipped slope array (no repeated network reads), masked to the parcel's own footprint.

In [ ]:
import numpy as np
from rasterio.features import geometry_mask


def parcel_mean_slope(geom, slope_arr, affine):
    minx, miny, maxx, maxy = geom.bounds
    col0, row1 = ~affine * (minx, miny)
    col1, row0 = ~affine * (maxx, maxy)
    row0, row1 = int(np.floor(row0)), int(np.ceil(row1))
    col0, col1 = int(np.floor(col0)), int(np.ceil(col1))
    row0, col0 = max(row0, 0), max(col0, 0)
    row1, col1 = min(row1, slope_arr.shape[0]), min(col1, slope_arr.shape[1])
    if row1 <= row0 or col1 <= col0:
        return np.nan
    window = slope_arr[row0:row1, col0:col1]
    window_transform = affine * affine.translation(col0, row0)
    mask = geometry_mask([geom], out_shape=window.shape, transform=window_transform, invert=True)
    values = window[mask & ~np.isnan(window)]
    return float(values.mean()) if values.size else np.nan


base_for_slope = base_for_slope.copy()
base_for_slope["mean_slope_pct"] = [
    parcel_mean_slope(geom, slope_pct, dem_transform) for geom in base_for_slope.geometry
]
print(f"parcels with no DEM coverage in this window: {base_for_slope['mean_slope_pct'].isna().sum()}")

slope_sensitivity_rows = []
slope_survivors_by_cutoff = {}
for cutoff in slope_cutoffs_pct:
    under = base_for_slope[base_for_slope["mean_slope_pct"] < cutoff]
    slope_survivors_by_cutoff[cutoff] = under
    slope_sensitivity_rows.append({
        "filter": "slope_cutoff_pct", "value": cutoff,
        "parcel_count": len(under), "area_ha": round(under["parcel_area_sqm"].sum() / 10_000, 2),
    })
pd.DataFrame(slope_sensitivity_rows)

## Primary funnel scenario

Carries the middle value from each of the three sensitivity-tested filters (100 m service distance, 20% building coverage, 20% slope) forward as the single scenario used for the funnel table and the network-distance stage below -- a judgment call about which values to feature, not a claim that these are the correct thresholds. The full sensitivity table (all tested values) is written separately so this choice is not the only number reported.

In [ ]:
final_survivors = slope_survivors_by_cutoff[primary_slope_cutoff_pct]
funnel_rows.append({
    "stage": f"5. within {primary_service_distance_m} m of water + sanitary mains",
    "parcel_count": len(service_survivors_by_distance[primary_service_distance_m]),
    "area_ha": round(service_survivors_by_distance[primary_service_distance_m]["parcel_area_sqm"].sum() / 10_000, 2),
})
funnel_rows.append({
    "stage": f"6. building coverage under {primary_building_coverage_cutoff:.0%}",
    "parcel_count": len(coverage_survivors_by_cutoff[primary_building_coverage_cutoff]),
    "area_ha": round(coverage_survivors_by_cutoff[primary_building_coverage_cutoff]["parcel_area_sqm"].sum() / 10_000, 2),
})
funnel_rows.append({
    "stage": f"7. mean DEM slope under {primary_slope_cutoff_pct}%",
    "parcel_count": len(final_survivors),
    "area_ha": round(final_survivors["parcel_area_sqm"].sum() / 10_000, 2),
})
parcel_funnel = pd.DataFrame(funnel_rows)
parcel_funnel

## Unit capacity: explicitly absent

research/inbox/ has no zoning bylaw density extraction (checked again this session). No unit-capacity column is added to any output; this note is the record of why, not a blank left to be misread as zero.

## Network distance: gondola base and downtown reference point

Builds a graph from the Centreline network (nodes = rounded line endpoints, edges weighted by length) and computes shortest-path distance from each Filter-7 survivor to the gondola base station (C072) and to the centroid of the Zoning layer's own "MU-1 - Downtown Zone" parcels -- both evidence-based reference points, not guessed landmarks.

In [ ]:
import networkx as nx

centreline = fetch_city_layer("Centreline_view")

def snap(coord, precision=1):
    return (round(coord[0], precision), round(coord[1], precision))

G = nx.Graph()
for geom in centreline.geometry:
    lines = geom.geoms if geom.geom_type == "MultiLineString" else [geom]
    for line in lines:
        coords = list(line.coords)
        for (x0, y0), (x1, y1) in zip(coords[:-1], coords[1:]):
            n0, n1 = snap((x0, y0)), snap((x1, y1))
            length = ((x1 - x0) ** 2 + (y1 - y0) ** 2) ** 0.5
            G.add_edge(n0, n1, weight=length)
components = sorted(nx.connected_components(G), key=len, reverse=True)
print(f"graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
      f"{len(components)} connected components (largest: {len(components[0])} nodes)")
# Centreline is real but genuinely fragmented (private lanes/driveways likely recorded as
# disconnected stubs); routing only makes sense within one connected piece, so the largest
# component is used from here on, not the whole graph -- a documented judgment call, not a
# silent one, and its effect (how far reference points sit from it) is reported below.
G_main = G.subgraph(components[0]).copy()

## Reference points

Gondola base from S050's own way geometry (C072, cross-checked against the DEM elevation already recorded in step 03's lift crosswalk); downtown reference from the live Zoning layer's MU-1 parcels' centroid (same query as Filter 1, C049).

In [ ]:
import pyproj
import shapely.geometry
from shapely.ops import transform as shp_transform

with open(f"{raw_dir}/S050_overpass_lifts_pistes.json", encoding="utf-8") as f:
    overpass_lifts = json.load(f)
gondola_lower = next(el for el in overpass_lifts["elements"] if el["id"] == 1361895818)
gondola_endpoints_4326 = [(pt["lon"], pt["lat"]) for pt in gondola_lower["geometry"][:1] + gondola_lower["geometry"][-1:]]
to_project = pyproj.Transformer.from_crs("EPSG:4326", project_crs, always_xy=True).transform
gondola_endpoints = [shp_transform(to_project, shapely.geometry.Point(pt)) for pt in gondola_endpoints_4326]
# CRS/units check on this reprojection (EPSG:4326 -> project_crs): UTM 11N eastings/northings
# for Revelstoke fall in the 400,000s/5,600,000s range, not the small lat/lon-scale numbers a
# skipped or reversed reprojection would leave behind.
assert all(4e5 < p.x < 5e5 and 5.5e6 < p.y < 5.7e6 for p in gondola_endpoints)
# Identify the base (bottom) endpoint by matching step 03's recorded bottom elevation (513.5 m, C072)
# rather than assuming an index; sample both endpoints against their own small DEM clip
# (the endpoints sit outside the filter-7 working extent, so slope_pct's window doesn't cover them).
gondola_dem = clip_dem(
    (min(p.x for p in gondola_endpoints) - 200, min(p.y for p in gondola_endpoints) - 200,
     max(p.x for p in gondola_endpoints) + 200, max(p.y for p in gondola_endpoints) + 200),
    project_crs, buffer_m=100,
)
gondola_arr = gondola_dem.values[0]
gondola_affine = gondola_dem.rio.transform()
gondola_elevs = [sample_at_point(gondola_arr, pt.x, pt.y, gondola_affine) for pt in gondola_endpoints]
gondola_base_point = gondola_endpoints[int(np.argmin(gondola_elevs))]
print(f"gondola endpoints' elevations: {gondola_elevs}, base point: {gondola_base_point}")

downtown_zone = zoning_live[zoning_live["zoning_code"] == "MU-1"]
downtown_point = downtown_zone.union_all().centroid
print(f"downtown reference point (MU-1 centroid): {downtown_point}")

## Shortest-path distance to each reference point

Each parcel's representative point snaps to the nearest graph node by brute-force nearest-neighbour search over node coordinates (networkx has no built-in spatial index, and scipy is not part of this project's environment). Only the largest connected component (see above) is used, so a parcel with no path is expected to be rare, not silently dropped -- see Checks below for exactly how many.

In [ ]:
node_array = np.array(list(G_main.nodes))

def nearest_node(point):
    d2 = (node_array[:, 0] - point.x) ** 2 + (node_array[:, 1] - point.y) ** 2
    return tuple(node_array[np.argmin(d2)]), float(np.sqrt(d2.min()))

gondola_node, gondola_snap_dist_m = nearest_node(gondola_base_point)
downtown_node, downtown_snap_dist_m = nearest_node(downtown_point)
print(f"gondola base snapped {gondola_snap_dist_m:.0f} m to the nearest main-network node "
      f"(large -- the resort base area itself has little/no Centreline coverage)")
print(f"downtown reference snapped {downtown_snap_dist_m:.0f} m to the nearest main-network node")

dist_to_gondola = nx.single_source_dijkstra_path_length(G_main, gondola_node, weight="weight")
dist_to_downtown = nx.single_source_dijkstra_path_length(G_main, downtown_node, weight="weight")

network_distances = final_survivors[["PID", "geometry"]].copy()
network_distances["rep_point"] = network_distances.geometry.representative_point()
snapped = network_distances["rep_point"].apply(nearest_node)
network_distances["nearest_node"] = snapped.apply(lambda t: t[0])
network_distances["own_snap_dist_m"] = snapped.apply(lambda t: t[1])
# Full distance = parcel's own snap-to-network gap + shortest path on the network + the
# reference point's own snap-to-network gap -- omitting either "last mile" would understate
# the true distance and can even (as first found) understate it below the straight-line
# distance, which is what the Checks cell below catches.
network_distances["network_dist_to_gondola_base_m"] = (
    network_distances["own_snap_dist_m"] + network_distances["nearest_node"].map(dist_to_gondola) + gondola_snap_dist_m
)
network_distances["network_dist_to_downtown_m"] = (
    network_distances["own_snap_dist_m"] + network_distances["nearest_node"].map(dist_to_downtown) + downtown_snap_dist_m
)
network_distances = network_distances.drop(columns=["rep_point", "nearest_node", "own_snap_dist_m", "geometry"])
print(f"parcels with no path to gondola base: {network_distances['network_dist_to_gondola_base_m'].isna().sum()}")
print(f"parcels with no path to downtown: {network_distances['network_dist_to_downtown_m'].isna().sum()}")
network_distances.describe()

## Write outputs

In [ ]:
os.makedirs(processed_dir, exist_ok=True)
parcel_funnel.to_csv(f"{processed_dir}/07_parcel_funnel.csv", index=False, encoding="utf-8")

funnel_sensitivity = pd.concat([
    pd.DataFrame(service_sensitivity_rows),
    pd.DataFrame(coverage_sensitivity_rows),
    pd.DataFrame(slope_sensitivity_rows),
], ignore_index=True)
funnel_sensitivity.to_csv(f"{processed_dir}/07_funnel_sensitivity.csv", index=False, encoding="utf-8")
network_distances.to_csv(f"{processed_dir}/07_network_distances.csv", index=False, encoding="utf-8")
print("wrote 07_parcel_funnel.csv, 07_funnel_sensitivity.csv, 07_network_distances.csv")

## Checks

Funnel count/area must be non-increasing at every stage; the starting count must match a fresh, independent re-query of ParcelMap BC's own count for the municipality; every network distance must be at least as long as the straight-line distance to the same reference point (a real geometric invariant, not just internal consistency); the downtown-to-gondola-base network distance is spot-checked against an independently-sourced figure (C074), not just checked for internal consistency; CRS confirmed metres after every reprojection.

In [ ]:
import requests

assert parcel_funnel["parcel_count"].is_monotonic_decreasing or parcel_funnel["parcel_count"].diff().dropna().le(0).all()
assert parcel_funnel["area_ha"].diff().dropna().le(0).all()

independent_count = requests.get(
    "https://openmaps.gov.bc.ca/geo/pub/WHSE_CADASTRE.PMBC_PARCEL_FABRIC_POLY_SVW/ows",
    params={
        "service": "WFS", "version": "2.0.0", "request": "GetFeature",
        "typeNames": "pub:WHSE_CADASTRE.PMBC_PARCEL_FABRIC_POLY_SVW",
        "resultType": "hits", "CQL_FILTER": "MUNICIPALITY='Revelstoke, City of'",
    },
    timeout=60,
).text
independent_total = int(independent_count.split('numberMatched="')[1].split('"')[0])
assert independent_total == len(parcels_raw), (independent_total, len(parcels_raw))

valid_idx = network_distances["network_dist_to_gondola_base_m"].dropna().index
rep_pts = final_survivors.loc[valid_idx].geometry.representative_point()
straight_line = rep_pts.distance(gondola_base_point)
assert (network_distances.loc[valid_idx, "network_dist_to_gondola_base_m"] >= straight_line.values - 1).all(), (
    "found a network distance shorter than the straight-line distance, which is geometrically impossible"
)

# Independent spot check (C074): a third-party ski-resort directory states the highway-exit-
# to-lift-entrance route is "8 km ... approx. 14 Minutes driving time". This step's own
# downtown-reference-to-gondola-base network distance uses different endpoints (MU-1 centroid,
# not the highway exit; gondola base station, not the lift entrance specifically), so it is
# checked as an order-of-magnitude match, not an exact one.
downtown_to_gondola_network_m = downtown_snap_dist_m + dist_to_gondola[downtown_node] + gondola_snap_dist_m
print(f"downtown-reference to gondola-base network distance: {downtown_to_gondola_network_m:.0f} m "
      f"(C074's independent figure for a similar but not identical route: ~8,000 m)")
assert 3_000 < downtown_to_gondola_network_m < 20_000, (
    "network distance is off by an order of magnitude from C074's independently-sourced figure"
)

assert dem.rio.crs.to_string() == project_crs
assert abs(abs(dem.rio.resolution()[0]) - 1.0) < 0.1
print("checks passed")

## Map: funnel stages and network reference points


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 9))
parcels.plot(ax=ax, color="#eeeeee", edgecolor="#cccccc", linewidth=0.2)
eligible.plot(ax=ax, color="#fdd0a2", edgecolor="none", alpha=0.6)
final_survivors.plot(ax=ax, color="#31a354", edgecolor="black", linewidth=0.3)
centreline.plot(ax=ax, color="grey", linewidth=0.5)
ax.scatter(*gondola_base_point.xy, color="blue", marker="^", s=100, label="gondola base", zorder=5)
ax.scatter(downtown_point.x, downtown_point.y, color="red", marker="*", s=150, label="downtown ref.", zorder=5)
ax.set_title("Parcel funnel: zoned-eligible (orange) vs. final survivors (green)")
ax.set_xlabel(f"Easting ({project_crs})")
ax.set_ylabel("Northing")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## Versions

In [ ]:
import importlib.metadata

for pkg in ["pandas", "geopandas", "shapely", "pyproj", "rasterio", "rioxarray", "networkx", "requests", "matplotlib"]:
    print(pkg, importlib.metadata.version(pkg))